# Dukascopy: millions of EUR/USD ticks

This notebook downloads Dukascopy's hourly BI5 tick files, decodes
bid/ask quotes, and sends every midpoint observation to one XY line.
The overview is M4-decimated to the viewport; wheel-zooming restores
detail from the canonical tick series.

The default is five calendar days beginning 2024-01-02. Change
`DUKASCOPY_START`, `DUKASCOPY_DAYS`, `DUKASCOPY_HOURS`, or
`DUKASCOPY_SYMBOL` to explore
a longer period. The price divisor below is correct for EUR/USD and
most five-decimal FX pairs.

**Source:** [Dukascopy Historical Data Export](https://www.dukascopy.com/swiss/english/marketwatch/historical/).

Install beside XY with `python -m pip install numpy requests xy`.


In [ ]:
import lzma
import os
import time as time_module
from datetime import UTC, date, datetime, time, timedelta
from pathlib import Path

import numpy as np
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data")) / "dukascopy"
DATA_DIR.mkdir(parents=True, exist_ok=True)

SYMBOL = os.getenv("DUKASCOPY_SYMBOL", "EURUSD").upper()
START = date.fromisoformat(os.getenv("DUKASCOPY_START", "2024-01-02"))
DAYS = int(os.getenv("DUKASCOPY_DAYS", "5"))
HOURS = int(os.getenv("DUKASCOPY_HOURS", "24"))
PRICE_DIVISOR = float(os.getenv("DUKASCOPY_PRICE_DIVISOR", "100000"))
REQUEST_DELAY = float(os.getenv("DUKASCOPY_REQUEST_DELAY", "0.2"))
if DAYS <= 0 or not 1 <= HOURS <= 24 or PRICE_DIVISOR <= 0 or REQUEST_DELAY < 0:
    raise ValueError("use positive days/divisor, 1-24 hours, and a non-negative delay")

SESSION = requests.Session()
SESSION.headers["User-Agent"] = "xy-real-world-notebook/1.0"
SESSION.mount(
    "https://",
    HTTPAdapter(
        max_retries=Retry(
            total=6,
            backoff_factor=1.0,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods={"GET"},
            respect_retry_after_header=True,
        )
    ),
)

TICK_DTYPE = np.dtype(
    [
        ("millisecond", ">u4"),
        ("ask", ">u4"),
        ("bid", ">u4"),
        ("ask_volume", ">f4"),
        ("bid_volume", ">f4"),
    ]
)

In [ ]:
def load_hour(day: date, hour: int) -> tuple[np.ndarray, np.ndarray]:
    relative = f"{SYMBOL}/{day.year}/{day.month - 1:02d}/{day.day:02d}/{hour:02d}h_ticks.bi5"
    url = f"https://datafeed.dukascopy.com/datafeed/{relative}"
    path = DATA_DIR / relative
    if not path.exists():
        response = SESSION.get(url, timeout=120)
        time_module.sleep(REQUEST_DELAY)
        if response.status_code == 404:
            return np.empty(0, dtype="datetime64[ms]"), np.empty(0)
        response.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(response.content)

    compressed = path.read_bytes()
    if not compressed:
        return np.empty(0, dtype="datetime64[ms]"), np.empty(0)
    ticks = np.frombuffer(lzma.decompress(compressed), dtype=TICK_DTYPE)
    hour_start = datetime.combine(
        day,
        time(hour=hour),
        tzinfo=UTC,
    )
    epoch_ms = int(hour_start.timestamp() * 1000)
    timestamps = (epoch_ms + ticks["millisecond"].astype(np.int64)).astype("datetime64[ms]")
    midpoint = (ticks["ask"].astype(np.float64) + ticks["bid"].astype(np.float64)) / (
        2 * PRICE_DIVISOR
    )
    return timestamps, midpoint


timestamp_parts = []
midpoint_parts = []
for day_offset in range(DAYS):
    day = START + timedelta(days=day_offset)
    for hour in range(HOURS):
        timestamps, midpoint = load_hour(day, hour)
        if midpoint.size:
            timestamp_parts.append(timestamps)
            midpoint_parts.append(midpoint)

if not midpoint_parts:
    raise RuntimeError("the selected interval returned no Dukascopy ticks")

timestamps = np.concatenate(timestamp_parts)
midpoint = np.concatenate(midpoint_parts)
print(f"{midpoint.size:,} {SYMBOL} ticks from {timestamps[0]} through {timestamps[-1]}")

In [ ]:
pair = f"{SYMBOL[:3]}/{SYMBOL[3:]}"
price_decimals = max(0, int(round(np.log10(PRICE_DIVISOR))))
price_span = float(np.ptp(midpoint))
raw_step = max(price_span / 6, 1 / PRICE_DIVISOR)
step_magnitude = 10 ** np.floor(np.log10(raw_step))
step_multiplier = next(
    candidate for candidate in (1, 2, 5, 10) if candidate >= raw_step / step_magnitude
)
price_step = float(step_multiplier * step_magnitude)
price_floor = np.floor(midpoint.min() / price_step) * price_step
price_ceiling = np.ceil(midpoint.max() / price_step) * price_step
price_ticks = np.arange(price_floor, price_ceiling + price_step / 2, price_step)
price_tick_labels = [f"{value:.4f}" for value in price_ticks]
session_rail_floor = price_ceiling + price_step * 0.02
session_rail_label_y = price_ceiling + price_step * 0.20

last_day_start = timestamps[-1].astype("datetime64[D]").astype("datetime64[ms]")
last_day_index = int(np.searchsorted(timestamps, last_day_start, side="left"))
day_open = float(midpoint[last_day_index])
day_change = float(midpoint[-1] - day_open)
day_change_percent = day_change / day_open * 100
pip_size = 0.01 if SYMBOL.endswith("JPY") else 0.0001
day_change_pips = day_change / pip_size
day_direction = "UP" if day_change > 0 else "DOWN" if day_change < 0 else "FLAT"
day_change_color = "#7ce8bc" if day_change >= 0 else "#ff9f8c"
last_timestamp_ms = int(timestamps[-1].astype("datetime64[ms]").astype(np.int64))

# Subtle regional-session bands make the intraday rhythm legible while
# preserving every tick in the interactive canonical series.
session_bands = []
session_boundaries = []
session_labels = []
session_specs = (
    (0, 7, "ASIA", "#17364a"),
    (7, 13, "LONDON", "#173b32"),
    (13, 21, "NEW YORK", "#3b3020"),
)
for day_offset in range(DAYS):
    session_start = datetime.combine(START + timedelta(days=day_offset), time())
    for start_hour, end_hour, label, color in session_specs:
        session_bands.append(
            xy.x_band(
                session_start + timedelta(hours=start_hour),
                session_start + timedelta(hours=end_hour),
                color=color,
                opacity=0.12,
            )
        )
        if day_offset == 0:
            label_color = {"ASIA": "#79b9d8", "LONDON": "#6fd4ad", "NEW YORK": "#d8b36f"}[label]
            session_labels.append(
                xy.text(
                    session_start + timedelta(hours=(start_hour + end_hour) / 2),
                    session_rail_label_y,
                    label,
                    dx=0,
                    dy=0,
                    color=label_color,
                    anchor="middle",
                    style={
                        "font_size": 10,
                        "font_weight": 750,
                        "letter_spacing": "0.08em",
                    },
                )
            )
    for boundary_hour in (7, 13, 21):
        session_boundaries.append(
            xy.vline(
                session_start + timedelta(hours=boundary_hour),
                color="#405268",
                width=1,
                opacity=0.65,
            )
        )

period_end = START + timedelta(days=DAYS - 1)
period_label = START.isoformat() if DAYS == 1 else f"{START.isoformat()} → {period_end.isoformat()}"

chart = xy.line_chart(
    *session_bands,
    *session_boundaries,
    xy.hline(
        session_rail_floor,
        color="#405268",
        width=1,
        opacity=0.72,
    ),
    *session_labels,
    xy.line(
        timestamps,
        midpoint,
        name=f"{pair} midpoint",
        color="#42e6a4",
        width=1.7,
    ),
    xy.line(
        [timestamps[0], timestamps[-1]],
        [midpoint[-1], midpoint[-1]],
        color="#8af7cb",
        width=1,
        opacity=0.62,
        dash=[5, 5],
    ),
    xy.text(
        timestamps[-1],
        float(midpoint[-1]),
        f"LAST · {midpoint[-1]:.{price_decimals}f}",
        dx=10,
        dy=0,
        color="#8af7cb",
        anchor="start",
        style={
            "background": "#071521",
            "border": "1px solid #42e6a480",
            "border_radius": 4,
            "font_size": 10.5,
            "font_weight": 750,
            "letter_spacing": "0.035em",
            "padding": "3px 7px",
            "vertical_align": "middle",
        },
    ),
    xy.text(
        0.097,
        0.965,
        f"{pair}  /  INTRADAY TICK TAPE",
        dx=0,
        dy=0,
        color="#8af7cb",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 20,
            "font_weight": 750,
            "letter_spacing": "0.045em",
            "vertical_align": "top",
        },
    ),
    xy.text(
        0.097,
        0.918,
        f"{period_label}  ·  {midpoint.size:,} TICKS  ·  DUKASCOPY HISTORICAL FEED",
        dx=0,
        dy=0,
        color="#8292a5",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 10.5,
            "font_weight": 600,
            "letter_spacing": "0.035em",
            "vertical_align": "top",
        },
    ),
    xy.text(
        0.882,
        0.959,
        f"DAY {day_direction}  {day_change_percent:+.2f}%  ·  {day_change_pips:+.1f} PIPS",
        dx=0,
        dy=0,
        color=day_change_color,
        anchor="end",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 12,
            "font_weight": 750,
            "letter_spacing": "0.025em",
            "vertical_align": "top",
        },
    ),
    xy.x_axis(
        label="TIME · UTC",
        type_="time",
        domain=(int(timestamps[0].astype(np.int64)), last_timestamp_ms),
        style={
            "grid_color": "#223044",
            "grid_width": 1,
            "grid_opacity": 0.45,
            "axis_color": "#55657a",
            "axis_width": 1.2,
            "tick_color": "#55657a",
            "tick_label_color": "#a8b5c5",
            "label_color": "#8af7cb",
        },
    ),
    xy.y_axis(
        label=f"{pair} · MID",
        label_offset=-28,
        domain=(price_floor - price_step * 0.2, price_ceiling + price_step * 0.38),
        tick_values=price_ticks,
        tick_labels=price_tick_labels,
        style={
            "grid_color": "#26354a",
            "grid_width": 1,
            "grid_dash": "dotted",
            "grid_opacity": 0.8,
            "axis_color": "#55657a",
            "axis_width": 1.2,
            "tick_color": "#55657a",
            "tick_label_color": "#c5d0dc",
            "label_color": "#8af7cb",
        },
    ),
    xy.legend(show=False),
    xy.tooltip(
        title=f"{pair} midpoint",
        format={"y": f".{price_decimals}f"},
    ),
    xy.interaction_config(
        crosshair=True,
        wheel_zoom=True,
        box_zoom=True,
        double_click_reset=True,
    ),
    xy.theme(
        background="#05080d",
        plot_background="#08111b",
        text_color="#c8d3df",
        grid_color="#26354a",
        axis_color="#55657a",
        crosshair_color="#f0c96a",
        selection_color="#42e6a4",
        selection_fill="#42e6a424",
    ),
    styles={
        "axis_title": {"font_weight": 700, "letter_spacing": "0.06em"},
        "tick_label": {
            "font_family": "ui-monospace, monospace",
            "font_variant_numeric": "tabular-nums",
        },
        "annotation_label": {"font_family": "ui-monospace, monospace"},
        "tooltip": {
            "background": "#071019",
            "color": "#d9e4ee",
            "border": "1px solid #2a4b4b",
            "border_radius": 3,
            "font_family": "ui-monospace, monospace",
        },
    },
    style={
        "border": "1px solid #172838",
        "font_family": "ui-monospace, SFMono-Regular, Menlo, monospace",
    },
    width=1150,
    height=560,
    padding=(98, 142, 68, 116),
)
payload = chart.figure().build_payload()[0]
print("render tier:", payload["traces"][0]["tier"])
chart